In [9]:
import os
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account

# 1. Definimos la ruta de la clave y el ID de tu proyecto
KEY_PATH = "../gcp-credentials.json"
PROJECT_ID = "fintech-credit-analytics"

# 2. Creamos las credenciales leyendo el archivo JSON
credentials = service_account.Credentials.from_service_account_file(KEY_PATH)

# 3. Inicializamos el cliente de BigQuery
client = bigquery.Client(credentials=credentials, project=PROJECT_ID)

print("[✓] Conexión establecida con BigQuery.")

[✓] Conexión establecida con BigQuery.


In [10]:
# Smoke Test: comprobamos formato y conectividad con solo 1000 filas
query_sample = f"""
    SELECT id_ingesta, ingested_at, source_system, payload_json
    FROM `{PROJECT_ID}.raw_vault.raw_solicitudes`
    LIMIT 1000
"""
df_sample = client.query(query_sample).to_dataframe()
print(f"[✓] Smoke test exitoso: {len(df_sample)} filas de muestra descargadas.")

[✓] Smoke test exitoso: 1000 filas de muestra descargadas.


In [11]:
import json

def cargar_tabla_cruda(nombre_tabla):
    query = f"""
        SELECT id_ingesta, ingested_at, source_system, payload_json
        FROM `{PROJECT_ID}.raw_vault.{nombre_tabla}`
    """
    df_raw = client.query(query).to_dataframe()
    # Desarmamos el JSON a columnas de Pandas
    payloads = df_raw["payload_json"].apply(json.loads).tolist()
    df_limpio = pd.DataFrame(payloads)
    df_limpio["ingested_at"] = df_raw["ingested_at"]
    return df_limpio

# Descarga de las 3 entidades del Data Warehouse
print("Descargando tablas completas desde BigQuery...")
df_clientes = cargar_tabla_cruda("raw_clientes")
df_solicitudes = cargar_tabla_cruda("raw_solicitudes")
df_pagos = cargar_tabla_cruda("raw_pagos")

print(f"[✓] df_clientes: {len(df_clientes):,} filas")
print(f"[✓] df_solicitudes: {len(df_solicitudes):,} filas")
print(f"[✓] df_pagos: {len(df_pagos):,} filas")

Descargando tablas completas desde BigQuery...
[✓] df_clientes: 3,290 filas
[✓] df_solicitudes: 8,000 filas
[✓] df_pagos: 129,834 filas


In [12]:
def inspeccion(df):
    display(df.head())

    display(df.info())

    display(df.describe())

    display(df.isna().sum())

    print(f"Número de filas duplicadas: {df.duplicated().sum()}")
    



In [13]:
inspeccion(df_solicitudes)


df_solicitudes['id_canal']

print("Tasas negativas:", (df_solicitudes['tasa_nominal_anual'] < 0).sum())




,id_solicitud,id_cliente,id_canal,monto_solicitado,monto_otorgado,tasa_nominal_anual,plazo_meses,fecha_desembolso,estado,created_at,ingested_at
0,1,370,3,3881000.0,3881000.0,-0.15,18,2026-04-04,APROBADO,2026-04-04T00:00:00,2026-10-16 08:42:42.508155+00:00
1,2,1358,4,2854000.0,2854000.0,-0.15,12,2025-04-08,APROBADO,2025-04-08T00:00:00,2026-10-16 08:42:42.508155+00:00
2,3,314,1,3666000.0,3666000.0,-0.15,36,2026-01-14,APROBADO,2026-01-14T00:00:00,2026-10-16 08:42:42.508155+00:00
3,4,1238,3,1173000.0,1173000.0,-0.15,24,2026-04-20,APROBADO,2026-04-20T00:00:00,2026-10-16 08:42:42.508155+00:00
4,5,2185,2,283000.0,283000.0,-0.15,24,2025-09-19,APROBADO,2025-09-19T00:00:00,2026-10-16 08:42:42.508155+00:00


<class 'pandas.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   id_solicitud        8000 non-null   int64              
 1   id_cliente          8000 non-null   int64              
 2   id_canal            8000 non-null   int64              
 3   monto_solicitado    8000 non-null   float64            
 4   monto_otorgado      8000 non-null   float64            
 5   tasa_nominal_anual  8000 non-null   float64            
 6   plazo_meses         8000 non-null   int64              
 7   fecha_desembolso    6769 non-null   str                
 8   estado              8000 non-null   str                
 9   created_at          8000 non-null   str                
 10  ingested_at         8000 non-null   datetime64[us, UTC]
dtypes: datetime64[us, UTC](1), float64(3), int64(4), str(3)
memory usage: 966.8 KB


None

,id_solicitud,id_cliente,id_canal,monto_solicitado,monto_otorgado,tasa_nominal_anual,plazo_meses
count,8000.00000,8000.000000,8000.000000,8.000000e+03,8.000000e+03,8000.000000,8000.000000
mean,4000.50000,1492.178000,3.511625,2.594252e+06,2.197445e+06,0.802764,19.209750
std,2309.54541,871.342964,31.517527,1.385721e+06,1.580181e+06,0.403480,10.286264
min,1.00000,1.000000,1.000000,2.010000e+05,0.000000e+00,-0.150000,6.000000
25%,2000.75000,728.750000,2.000000,1.397000e+06,7.450000e+05,0.639200,12.000000
50%,4000.50000,1483.500000,3.000000,2.586500e+06,2.170500e+06,0.879000,18.000000
75%,6000.25000,2248.250000,4.000000,3.793000e+06,3.571000e+06,1.108175,24.000000
max,8000.00000,3000.000000,999.000000,4.999000e+06,4.999000e+06,1.350000,36.000000


id_solicitud             0
id_cliente               0
id_canal                 0
monto_solicitado         0
monto_otorgado           0
tasa_nominal_anual       0
plazo_meses              0
fecha_desembolso      1231
estado                   0
created_at               0
ingested_at              0
dtype: int64

Número de filas duplicadas: 0
Tasas negativas: 5


### Hallazgos de Auditoría: `raw_solicitudes`
* **Anomalía de Regla de Negocio (Tasas Negativas):** Se detectan 5 registros con `tasa_nominal_anual < 0` (mínimo en `-0.15`). Justifica la implementación de un test singular en dbt (`assert_tasa_interes_positiva.sql`).
* **Valores Nulos (`fecha_desembolso`):** Existen 1.231 registros nulos. Corresponden a solicitudes con `estado != 'APROBADO'` (rechazadas), por lo que la ausencia de fecha es coherente con el ciclo de vida del crédito.
* **Integridad Referencial (`id_canal`):** Se identifican registros con valor `999`, inexistente en el catálogo de canales válidos (1 a 4). Justifica el mapeo a clave comodín `-1` (*Canal Desconocido*) en la dimensión `dim_canales` bajo modelado Kimball.

In [14]:
inspeccion(df_clientes)


,id_cliente,nombre,apellido,email,provincia,ingresos_mensuales,score_bancario,updated_at,ingested_at
0,1,Agustina,Gomez,ivanmartinez@example.net,Mendoza,745126.31,677,2024-07-11T00:00:00,2026-10-16 08:42:38.513288+00:00
1,2,Matias,Gomez,bautista51@example.net,Neuquén,2683060.76,421,2024-09-11T00:00:00,2026-10-16 08:42:38.513288+00:00
2,3,Rosario,Fernandez,gutierrezsantino@example.com,CABA,498526.31,566,2024-03-26T00:00:00,2026-10-16 08:42:38.513288+00:00
3,4,Carmela,Garcia,francescabenitez@example.com,Neuquén,2266258.06,608,2025-02-23T00:00:00,2026-10-16 08:42:38.513288+00:00
4,5,Nicolas,Escobar,avilajazmin@example.org,Salta,2574107.42,682,2024-07-17T00:00:00,2026-10-16 08:42:38.513288+00:00


<class 'pandas.DataFrame'>
RangeIndex: 3290 entries, 0 to 3289
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   id_cliente          3290 non-null   int64              
 1   nombre              3290 non-null   str                
 2   apellido            3290 non-null   str                
 3   email               3290 non-null   str                
 4   provincia           3290 non-null   str                
 5   ingresos_mensuales  3290 non-null   float64            
 6   score_bancario      3290 non-null   int64              
 7   updated_at          3290 non-null   str                
 8   ingested_at         3290 non-null   datetime64[us, UTC]
dtypes: datetime64[us, UTC](1), float64(1), int64(2), str(5)
memory usage: 437.8 KB


None

,id_cliente,ingresos_mensuales,score_bancario
count,3290.000000,3.290000e+03,3290.000000
mean,1501.024012,1.986402e+06,603.211550
std,865.069428,9.076444e+05,143.394362
min,1.000000,3.967227e+05,300.000000
25%,759.250000,1.220403e+06,481.000000
50%,1499.500000,1.993571e+06,604.000000
75%,2249.750000,2.733083e+06,726.000000
max,3000.000000,4.644995e+06,850.000000


id_cliente            0
nombre                0
apellido              0
email                 0
provincia             0
ingresos_mensuales    0
score_bancario        0
updated_at            0
ingested_at           0
dtype: int64

Número de filas duplicadas: 0


### Hallazgos de Auditoría: `raw_clientes`
* **Completitud:** Cero valores nulos en todas las columnas demográficas y financieras.
* **Distribución de Atributos:** Rango de `score_bancario` entre 300 y 850 (escala estándar crediticia) con media en 603.
* **Estructura de Claves:** No existen filas 100% idénticas, pero la cardinalidad de `id_cliente` (3.000 únicos sobre 3.290 filas) evidencia ~290 actualizaciones de estado. Justifica el modelado de `dim_clientes` como dimensión de cambio lento (SCD Tipo 2).

In [15]:
inspeccion(df_pagos)

,id_pago,id_solicitud,numero_cuota,fecha_vencimiento,fecha_pago_real,monto_cuota,dias_mora,estado_pago,ingested_at
0,1,1,1,2026-05-04,2026-05-04,562928.27,0,PAGADO,2026-10-16 08:42:47.729193+00:00
1,2,1,2,2026-06-03,2026-06-03,562928.27,0,PAGADO,2026-10-16 08:42:47.729193+00:00
2,3,1,3,2026-07-03,NaN,562928.27,90,MORA,2026-10-16 08:42:47.729193+00:00
3,4,1,4,2026-08-02,2026-08-14,562928.27,12,PAGADO,2026-10-16 08:42:47.729193+00:00
4,5,1,5,2026-09-01,2026-09-01,562928.27,0,PAGADO,2026-10-16 08:42:47.729193+00:00


<class 'pandas.DataFrame'>
RangeIndex: 129834 entries, 0 to 129833
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype              
---  ------             --------------   -----              
 0   id_pago            129834 non-null  int64              
 1   id_solicitud       129834 non-null  int64              
 2   numero_cuota       129834 non-null  int64              
 3   fecha_vencimiento  129834 non-null  str                
 4   fecha_pago_real    50861 non-null   str                
 5   monto_cuota        129834 non-null  float64            
 6   dias_mora          129834 non-null  int64              
 7   estado_pago        129834 non-null  str                
 8   ingested_at        129834 non-null  datetime64[us, UTC]
dtypes: datetime64[us, UTC](1), float64(1), int64(4), str(3)
memory usage: 11.6 MB


None

,id_pago,id_solicitud,numero_cuota,monto_cuota,dias_mora
count,129834.000000,129834.000000,129834.000000,1.298340e+05,129834.000000
mean,64917.500000,4008.068534,12.858103,3.397581e+05,20.177111
std,37479.991762,2289.763403,9.134533,2.163895e+05,75.672222
min,1.000000,1.000000,1.000000,1.572888e+04,0.000000
25%,32459.250000,2018.000000,5.000000,1.694283e+05,0.000000
50%,64917.500000,4004.000000,11.000000,3.156820e+05,0.000000
75%,97375.750000,5986.000000,18.000000,4.716365e+05,2.000000
max,129834.000000,7999.000000,36.000000,1.382645e+06,608.000000


id_pago                  0
id_solicitud             0
numero_cuota             0
fecha_vencimiento        0
fecha_pago_real      78973
monto_cuota              0
dias_mora                0
estado_pago              0
ingested_at              0
dtype: int64

Número de filas duplicadas: 0


### Hallazgos de Auditoría: `raw_pagos`
* **Completitud y Nulos:** Se observan 78.973 nulos en `fecha_pago_real`. Corresponden a cuotas futuras no devengadas o cuotas vencidas en estado `MORA`.
* **Tipado de Datos:** Las columnas temporales (`fecha_vencimiento`, `fecha_pago_real`) ingresaron como cadenas de texto (`object/string`), requiriendo casteo explícito a tipo fecha para el cálculo de ventanas de mora.
* **Métricas de Mora:** Se detectan atrasos máximos de hasta 608 días, variable clave a consolidar a nivel préstamo.

In [16]:
df_mora_prestamo = df_pagos.groupby("id_solicitud").agg(
    dias_mora_max=("dias_mora", "max"),
    cuotas_totales=("numero_cuota", "count"),
    cuotas_en_mora=("estado_pago", lambda x: (x == "MORA").sum())
).reset_index()

In [ ]:
#Clientes: dejamos solo 1 versión por cliente (la última ingresada)
df_clientes_unicos = df_clientes.sort_values("ingested_at").drop_duplicates(subset=["id_cliente"], keep="last")

#Solicitudes: solo aprobadas (las rechazadas nunca llegaron a tener cuotas ni pagos)
df_aprobadas = df_solicitudes[df_solicitudes["estado"] == "APROBADO"].copy()

#Cruce 1: Préstamos aprobados + Resumen de Mora (Relación 1 a 1 exacta)
df_analisis = df_aprobadas.merge(df_mora_prestamo, on="id_solicitud", how="left")

#Cruce 2: Pegamos el scoring del cliente (Relación N a 1 limpia)
df_analisis = df_analisis.merge(
    df_clientes_unicos[["id_cliente", "score_bancario", "ingresos_mensuales", "provincia"]],
    on="id_cliente",
    how="inner"
)

#mputación
df_analisis["dias_mora_max"] = df_analisis["dias_mora_max"].fillna(0)

In [18]:
columnas_numericas = [
    "score_bancario", 
    "ingresos_mensuales", 
    "monto_otorgado", 
    "tasa_nominal_anual", 
    "dias_mora_max"
]

matriz_corr = df_analisis[columnas_numericas].corr()
display(matriz_corr.round(3))

,score_bancario,ingresos_mensuales,monto_otorgado,tasa_nominal_anual,dias_mora_max
score_bancario,1.000,0.030,-0.016,-0.008,0.000
ingresos_mensuales,0.030,1.000,-0.004,0.013,0.000
monto_otorgado,-0.016,-0.004,1.000,-0.004,-0.012
tasa_nominal_anual,-0.008,0.013,-0.004,1.000,-0.017
dias_mora_max,0.000,0.000,-0.012,-0.017,1.000


## 4. Conclusiones del Diagnóstico y Especificaciones para dbt

### 4.1. Análisis Exploratorio y Validación de Datos (EDA)
* **Correlaciones Lineales Cercanas a Cero:** Las correlaciones de Pearson entre variables de scoring, ingresos, monto y mora arrojan coeficientes próximos a 0.0 (por ejemplo, `score_bancario` vs `dias_mora_max` en 0.0). Esto confirma la naturaleza estocástica del generador de datos sintéticos y demuestra la ausencia de una relación lineal directa trivial, justificando el diseño de métricas agregadas discretas y cohortes de riesgo en la capa analítica.

---

### 4.2. Hallazgos de Calidad de Datos (Reglas para Capa Bronze -> Silver)
1. **Regla de Negocio - Tasas Inválidas:**
   * Detección de 5 solicitudes con `tasa_nominal_anual < 0` (valores negativos en `-0.15`).
   * **Acción en dbt:** Creación de un singular test (`tests/assert_tasa_interes_positiva.sql`) para capturar anomalías y aislamiento/filtrado en el modelo de staging de solicitudes.
2. **Integridad Referencial - Canales Huérfanos:**
   * Detección de 8 registros con `id_canal = 999`, ausente en el catálogo dimensional de canales (1 al 4).
   * **Acción en dbt:** Mapeo de valores huérfanos a una clave comodín `-1` (*Canal Desconocido*) en `dim_canales` bajo modelado dimensional Kimball para preservar los importes en balance.
3. **Gestión de Historial - Dimensión Clientes:**
   * Detección de ~290 registros duplicados a nivel `id_cliente` producto de actualizaciones de score e ingresos.
   * **Acción en dbt:** Modelado de `dim_clientes` como dimensión de cambio lento (SCD Tipo 2) con claves subrogadas y banderas de registro actual (`es_actual`) para evitar duplicaciones en hechos (*fan-out*).
4. **Casteo de Metadatos y Tipos:**
   * Conversión explícita de campos temporales (`fecha_vencimiento`, `fecha_pago_real`) ingresados como cadenas de texto a tipo `DATE` en `stg_pagos`.
   * Los 1.231 nulos en `fecha_desembolso` se conservan como válidos por corresponder a solicitudes rechazadas.

---

### 4.3. Estrategia de Modelado (Capa Intermediate / Gold)
* Construcción de la tabla de hechos `fct_prestamos` consolidando las métricas de comportamiento calculadas a nivel solicitud (`dias_mora_max`, `cuotas_en_mora`, y clasificación de atraso) para consumo directo en BI.